# **Setup and load data**

load the reports, our rule labels, and the 58 gold studies. Gold studies are kept out of training.

In [4]:
import os, glob, re, time, unicodedata
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
_c = (glob.glob("/kaggle/input/datasets/*/rsna-knee-pseudo-labels/pseudo_labels_v2.csv")
      + glob.glob("/kaggle/input/rsna-knee-pseudo-labels/pseudo_labels_v2.csv"))
assert _c, "pseudo_labels_v2.csv not found - check the dataset is added under Input"
PL = _c[0]
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

train = pd.read_csv(f"{ROOT}/train.csv")
pl = pd.read_csv(PL)
assert (pl.StudyInstanceUID.values == train.StudyInstanceUID.values).all()
train["lang"] = pl.lang.values
train["covered"] = pl.covered.astype(bool).values
RULE = pl[LABELS].copy()                       # rule labels, NaN where language not covered
train["is_gold"] = train[LABELS].notna().all(axis=1)

def norm(t):
    t = unicodedata.normalize("NFKC", str(t)).replace("\u0130", "i")
    return t.lower().replace("\u0307", "")
train["text"] = train.Report.map(norm)

fit_idx = train.index[train.covered & ~train.is_gold]   # training set (rule labels)
gold_idx = train.index[train.is_gold]                   # evaluation set (real labels)
Y_fit = RULE.loc[fit_idx].values.astype(np.float32)
print("train:", len(fit_idx), "| gold:", len(gold_idx))

train: 3180 | gold: 58


# **Evaluation helpers**

one scoring function so every method is judged the same way on the same 58 studies.

In [5]:
RESULTS, PRED = {}, {}
Yg = train.loc[gold_idx, LABELS].astype(int).values
cov_g = train.loc[gold_idx, "covered"].values

def summarize(name, P):
    P = np.asarray(P, dtype=float); out = {"method": name}
    for tag, m in [("all58", np.ones(len(Yg), bool)), ("covered", cov_g), ("uncovered", ~cov_g)]:
        out[f"F1_{tag}"] = round(float(np.mean([f1_score(Yg[m, j], (P[m, j] >= 0.5).astype(int), zero_division=0)
                                                for j in range(12)])), 3)
    aucs = [roc_auc_score(Yg[:, j], P[:, j]) for j in range(12) if len(set(Yg[:, j])) == 2]
    out["macroAUC_all58"] = round(float(np.mean(aucs)), 3)
    RESULTS[name] = out; return out

def per_finding(P):
    P = np.asarray(P, dtype=float)
    return [round(f1_score(Yg[:, j], (P[:, j] >= 0.5).astype(int), zero_division=0), 2) for j in range(12)]

# **Method A, rule labeler**

our hand-written rules are the baseline to beat.

In [6]:
PRED["rules_v2"] = RULE.loc[gold_idx].fillna(0).values
print(summarize("rules_v2", PRED["rules_v2"]))

{'method': 'rules_v2', 'F1_all58': 0.587, 'F1_covered': 0.695, 'F1_uncovered': 0.0, 'macroAUC_all58': 0.698}


# **Method B, TF-IDF + logistic regression**
classical text features (character n-grams) with a simple classifier, trained on the rule labels.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=200000)
Xfit = vec.fit_transform(train.loc[fit_idx, "text"]); Xgold = vec.transform(train.loc[gold_idx, "text"])
P = np.zeros((len(gold_idx), 12))
for j in range(12):
    clf = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced")
    clf.fit(Xfit, Y_fit[:, j]); P[:, j] = clf.predict_proba(Xgold)[:, 1]
PRED["tfidf_lr"] = P
print(summarize("tfidf_lr", P))

{'method': 'tfidf_lr', 'F1_all58': 0.547, 'F1_covered': 0.629, 'F1_uncovered': 0.066, 'macroAUC_all58': 0.713}


# **Transformer setup**

a multilingual encoder-only transformer (DistilBERT). This cell defines the tokenizer, data loader, weighted-loss training loop and prediction function. It's the updated version.

In [9]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
MAX_LEN, BS, EPOCHS, LR = 384, 16, 8, 6e-5
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, texts, Y=None):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN); self.Y = Y
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
        if self.Y is not None: d["labels"] = self.Y[i]
        return d

def collate(batch):
    labels = [b.pop("labels") for b in batch] if "labels" in batch[0] else None
    out = tok.pad(batch, return_tensors="pt")
    if labels is not None: out["labels"] = torch.tensor(np.stack(labels), dtype=torch.float32)
    return out

def train_model(texts, Y):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=12, problem_type="multi_label_classification").to(dev)
    pos = np.clip(Y.mean(0), 0.02, 0.98)       # weight rare findings more
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.clip((1 - pos) / pos, 1, 8), dtype=torch.float32).to(dev))
    dl = DataLoader(DS(texts, Y), batch_size=BS, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
    for ep in range(EPOCHS):
        model.train(); tot, t0 = 0.0, time.time()
        for b in dl:
            b = {k: v.to(dev) for k, v in b.items()}
            labels = b.pop("labels")
            with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                logits = model(**b).logits
            loss = loss_fn(logits.float(), labels)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        print(f"epoch {ep + 1}/{EPOCHS} | train loss {tot / len(dl):.4f} | {time.time() - t0:.0f}s")
    return model

@torch.no_grad()
def predict(model, texts):
    model.eval(); out = []
    for b in DataLoader(DS(texts), batch_size=32, collate_fn=collate):
        b = {k: v.to(dev) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            out.append(torch.sigmoid(model(**b).logits.float()).cpu().numpy())
    return np.concatenate(out)

# **Train the transformer and score it on gold**

train on 90% of our rule-labeled studies, check agreement on the other 10% (sanity only), then score on the 58 gold studies. Takes about 4 to 5 minutes.

In [10]:
perm = np.random.RandomState(SEED).permutation(len(fit_idx)); nv = len(perm) // 10
va, tr = perm[:nv], perm[nv:]
model = train_model(train.loc[fit_idx[tr], "text"].values, Y_fit[tr])
pv = predict(model, train.loc[fit_idx[va], "text"].values)
agree = np.mean([f1_score(Y_fit[va][:, j], (pv[:, j] >= 0.5).astype(int), zero_division=0) for j in range(12)])
print(f"agreement with rules on held-out studies (mean F1): {agree:.3f}  <- should be well above 0.48 now")
PRED["transformer"] = predict(model, train.loc[gold_idx, "text"].values)
print(summarize("transformer", PRED["transformer"]))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/8 | train loss 0.9838 | 32s
epoch 2/8 | train loss 0.7455 | 33s
epoch 3/8 | train loss 0.5839 | 33s
epoch 4/8 | train loss 0.4780 | 33s
epoch 5/8 | train loss 0.3830 | 33s
epoch 6/8 | train loss 0.3071 | 33s
epoch 7/8 | train loss 0.2482 | 33s
epoch 8/8 | train loss 0.2148 | 33s
agreement with rules on held-out studies (mean F1): 0.794  <- should be well above 0.48 now
{'method': 'transformer', 'F1_all58': 0.571, 'F1_covered': 0.616, 'F1_uncovered': 0.299, 'macroAUC_all58': 0.713}


# 7: **a - Comparison table (this is the result for the report)**

all three methods side by side, per finding and per language.

In [11]:
res = pd.DataFrame(RESULTS.values()).set_index("method"); print(res.to_string())
pf = pd.DataFrame({m: per_finding(P) for m, P in PRED.items()}, index=LABELS)
print("\nper-finding F1 on gold:\n", pf.to_string())
lg = train.loc[gold_idx, "lang"].values
acc = pd.DataFrame({m: ((np.asarray(P) >= 0.5).astype(int) == Yg).mean(axis=1) for m, P in PRED.items()}); acc["lang"] = lg
print("\nlabel accuracy by language:\n", acc.groupby("lang").mean().round(2).assign(n=acc.groupby("lang").size()).to_string())
res.to_csv("/kaggle/working/text_model_results.csv"); pf.to_csv("/kaggle/working/text_model_per_finding_f1.csv")

             F1_all58  F1_covered  F1_uncovered  macroAUC_all58
method                                                         
rules_v2        0.587       0.695         0.000           0.698
tfidf_lr        0.547       0.629         0.066           0.713
transformer     0.571       0.616         0.299           0.713

per-finding F1 on gold:
                   rules_v2  tfidf_lr  transformer
ACL                   0.79      0.70         0.68
MCL                   0.61      0.46         0.40
Medial Meniscus       0.72      0.67         0.55
Lateral Meniscus      0.78      0.65         0.67
Medial OA             0.48      0.41         0.56
Lateral OA            0.57      0.48         0.52
PF OA                 0.36      0.46         0.47
Effusion              0.64      0.59         0.72
Synovitis             0.49      0.51         0.57
Baker's               0.55      0.54         0.67
Contusion             0.55      0.55         0.58
Fracture              0.52      0.55         0.48

lab

# **7 - b : quick blend check**

check whether averaging rules and transformer ranks gold studies better than either alone.

In [12]:
Pt, Pr, Pf = (np.asarray(PRED[k]) for k in ("transformer", "rules_v2", "tfidf_lr"))
blend = np.where(cov_g[:, None], 0.5 * Pr + 0.5 * Pt, Pt)
def mauc(P): return round(float(np.mean([roc_auc_score(Yg[:, j], P[:, j]) for j in range(12) if len(set(Yg[:, j])) == 2])), 3)
print("macro AUC on gold | rules:", mauc(Pr), "| tfidf:", mauc(Pf), "| transformer:", mauc(Pt), "| blend:", mauc(blend))

macro AUC on gold | rules: 0.698 | tfidf: 0.713 | transformer: 0.713 | blend: 0.743


# **8 : Cross-fitted pseudo-labels v3 (run only if the transformer wins in Cell 7)**

each study gets a prediction from a model that never saw it. Takes about 15 minutes with 8 epochs.

In [13]:
K = 3
folds = np.array_split(np.random.RandomState(SEED + 1).permutation(len(fit_idx)), K)
P_all = np.zeros((len(train), 12)); others = train.index[~train.index.isin(fit_idx)]
for k in range(K):
    te = folds[k]; trn = np.concatenate([folds[i] for i in range(K) if i != k])
    m = train_model(train.loc[fit_idx[trn], "text"].values, Y_fit[trn])
    P_all[fit_idx[te]] = predict(m, train.loc[fit_idx[te], "text"].values)
    P_all[others] += predict(m, train.loc[others, "text"].values) / K
    del m; torch.cuda.empty_cache()
out = train[["StudyInstanceUID", "lang", "covered", "is_gold"]].copy()
for j, l in enumerate(LABELS):
    out[f"p_{l}"] = P_all[:, j]; out[f"r_{l}"] = RULE[l].values
out.to_csv("/kaggle/working/pseudo_labels_v3.csv", index=False); print("saved", out.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/8 | train loss 0.9996 | 24s
epoch 2/8 | train loss 0.8107 | 24s
epoch 3/8 | train loss 0.6580 | 24s
epoch 4/8 | train loss 0.5361 | 24s
epoch 5/8 | train loss 0.4468 | 24s
epoch 6/8 | train loss 0.3705 | 24s
epoch 7/8 | train loss 0.3152 | 24s
epoch 8/8 | train loss 0.2781 | 24s


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/8 | train loss 1.0014 | 24s
epoch 2/8 | train loss 0.8069 | 24s
epoch 3/8 | train loss 0.6354 | 24s
epoch 4/8 | train loss 0.5099 | 24s
epoch 5/8 | train loss 0.4115 | 24s
epoch 6/8 | train loss 0.3402 | 24s
epoch 7/8 | train loss 0.2849 | 24s
epoch 8/8 | train loss 0.2491 | 24s


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/8 | train loss 0.9924 | 24s
epoch 2/8 | train loss 0.8109 | 24s
epoch 3/8 | train loss 0.6527 | 24s
epoch 4/8 | train loss 0.5139 | 24s
epoch 5/8 | train loss 0.4281 | 24s
epoch 6/8 | train loss 0.3503 | 24s
epoch 7/8 | train loss 0.2942 | 24s
epoch 8/8 | train loss 0.2615 | 24s
saved (4407, 28)


# **final soft labels for the image model**

for covered studies, average the rule label and the transformer's out-of-fold probability. For the other-language studies, use the transformer's probability alone.

In [14]:
v3 = pd.read_csv("/kaggle/working/pseudo_labels_v3.csv")
soft = pd.DataFrame({"StudyInstanceUID": v3.StudyInstanceUID, "lang": v3.lang, "is_gold": v3.is_gold})
for l in LABELS:
    soft[l] = np.where(v3.covered, 0.5 * v3[f"p_{l}"] + 0.5 * v3[f"r_{l}"].fillna(0), v3[f"p_{l}"])
soft.to_csv("/kaggle/working/soft_labels_final.csv", index=False)
print(soft[LABELS].mean().round(3).to_string())
print("rows:", len(soft), "| non-gold training rows:", int((~soft.is_gold).sum()))

ACL                 0.142
MCL                 0.155
Medial Meniscus     0.330
Lateral Meniscus    0.174
Medial OA           0.234
Lateral OA          0.169
PF OA               0.317
Effusion            0.445
Synovitis           0.151
Baker's             0.271
Contusion           0.220
Fracture            0.068
rows: 4407 | non-gold training rows: 4349
